[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# JSON in a Response


## What you will be able to do

Find your way around a JSON response you have not seen before: see its whole shape, reach any field
through nested objects and lists, and handle a field that is missing or `null` without the program
stopping.


## The idea

### The problem

The **Query Parameters** notebook reached into Open-Meteo's response with names it already knew:
`daily`, then a variable. Most responses are deeper than that, and less regular. The practice API's
`/network` document holds a list of stations. Each station holds a location object, a list of
instruments and a status object, and the status holds a list of its own. Not every station has
every field: Svalbard's location has no elevation, two instruments have a `last_calibrated` of
`null`, and Tromso's whole status is `null`.

Code written by looking at the first station works for the first station.
`station["location"]["elevation_m"]` raises `KeyError` when a loop reaches Svalbard, and
`station["status"]["active"]` raises `TypeError` at Tromso. A test that uses only the first station
passes, and the same code stops at the third or the fourth.

So reaching the field you want takes three things: knowing the shape of the whole response rather
than of one item, reaching into it a level at a time, and deciding in advance what the program does
where a level is missing or `null`.

### What JSON in a response is

> The body of a JSON response holds one JSON value, almost always an **object** or an **array**.
> An object holds named members and an array holds values in order, and either can hold more objects
> and arrays, which makes the response **nested**. `response.json()` parses the body into Python, as
> the **JSON on Disk** notebook did for files: objects become dictionaries, arrays become lists, and
> `null` becomes `None`. A **path** names one place inside the value, with a name for each object
> and a position for each array, as in `stations[2].location`.

### Why it works that way

- **JSON carries no promise about its shape.** Nothing in a body says which fields must be present
  or what type each one holds. An API's documentation says it, if anything does, and the **Schemas
  and Validation** notebook turns that promise into a check.
- **Missing and `null` are two different answers.** A missing field says nothing at all, and a field
  whose value is `null` says there is no value. APIs use both, sometimes for the same idea, and
  `dict.get` treats them alike unless the code checks.
- **An object is reached by name, and an array by position or by a loop.** A path is a chain of
  those steps, and each step has to suit the kind of value it lands on, which is what most of the
  errors in this notebook are about.
- **Dates are text.** JSON has no date type, so a date arrives as a string, usually in ISO 8601, and
  becomes a date only when the code parses it.
- **The same data comes in different shapes.** The network document sends rows: an object for each
  station. Open-Meteo sends columns: a list for each variable, matched by position. Columns repeat no
  names, so they are smaller, and rows are easier to filter and sort.

### Where you will meet this

Every JSON API nests. GitHub's API puts an `owner` object inside every repository, Stripe wraps a
list of results in an object with `data` and `has_more` members, and GeoJSON, which mapping services
send, writes a shape's coordinates as arrays inside arrays. Postman and a browser's developer tools
show a JSON response as a tree you can fold open, and `jq` reaches into JSON from the command line.
The paths in this notebook are written the way JSONPath writes them, as in `$.stations[*].name`;
JSONPath is a query language for JSON, standardized in 2024 as RFC 9535.

### What this notebook covers

- A response's JSON as Python values, and one item laid out with `json.dumps`
- An outline of every path in a document, with how often each appears and with which types
- Reaching in a level at a time, and finding an item by its id
- A field that is missing, a field that is `null`, and `dig`, which handles both along a path
- Dates in a document, parsed with `fromisoformat`
- Columns to rows, for Open-Meteo's `daily`
- A calibration report for the whole network
- Five errors, from a field one station lacks to a body with a JSON value on every line

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

network = requests.get("http://127.0.0.1:8765/network", timeout=10).json()

for station in network["stations"]:
    elevation = station["location"].get("elevation_m", "unknown")
    print(station["name"], len(station["instruments"]), "instruments, elevation", elevation)
```

```
Bergen 2 instruments, elevation 12
Oslo 3 instruments, elevation 94
Svalbard 2 instruments, elevation unknown
Tromso 2 instruments, elevation 100
```

An object holding a list, objects inside the list, and a field one station does not have, handled
with `get`: most of this notebook, in five lines.


## Setup

Nine imports, the last of them the practice API.

- `requests` sends every request, and its `json()` parses each response's body
- `json` lays a value out with `dumps`, and parses a single line with `loads`
- `Counter` counts the paths found in a document
- `date` and `datetime` turn the dates in a document from text into values
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `open_meteo()` returns Open-Meteo's address, or the address of the practice API's recording of
  it when Open-Meteo is not answering

If Open-Meteo stops answering while you work through the notebook, run this cell again: it checks
again, and the Open-Meteo cells switch to the recording.


In [1]:
import importlib
import json
import sys
import urllib.request
from collections import Counter
from datetime import date, datetime
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("The practice API is running at", BASE)
print("Open-Meteo's archive is at", OPEN_METEO)


The practice API is running at http://127.0.0.1:8765
Open-Meteo's archive is at https://archive-api.open-meteo.com/v1/archive


## Worked examples

### A response's JSON, as Python values

`/network` is a document the practice API serves for this notebook: a made-up network of the four
stations, shaped the way real API responses are shaped. `json()` parses its body:


In [2]:
response = requests.get(f"{BASE}/network", timeout=10)
network = response.json()

print(response.headers["Content-Type"])
print(type(network).__name__, list(network))
print(type(network["stations"]).__name__, len(network["stations"]))


application/json
dict ['name', 'updated', 'stations']
list 4


The body was one JSON object, so `json()` returned a dictionary, and its `stations` member, an array,
became a list of four. Here is a value of each type from inside it:


In [3]:
bergen = network["stations"][0]
examples = [bergen, bergen["instruments"], bergen["name"], bergen["location"]["elevation_m"],
            bergen["location"]["latitude"], bergen["status"]["active"], network["stations"][3]["status"]]

for value in examples:
    print(f"{type(value).__name__:<8} {repr(value)[:72]}")


dict     {'id': 'bergen', 'name': 'Bergen', 'location': {'latitude': 60.39, 'long
list     [{'kind': 'thermometer', 'installed': '2018-06-01', 'last_calibrated': '
str      'Bergen'
int      12
float    60.39
bool     True
NoneType None


Objects became dictionaries and arrays became lists. `12` became an `int` and `60.39` a `float`,
`true` became `True`, and Tromso's `status`, `null` in the body, became `None`. A dictionary this
deep prints on one long line, cut short here, so the next section lays one out.

### One item laid out: json.dumps

`json.dumps` with `indent`, from the **JSON on Disk** notebook, writes a value back out as JSON, with
each level of nesting indented further. It is the quickest way to read a response you have not seen
before. Here is Svalbard:


In [4]:
print(json.dumps(network["stations"][2], indent=2))


{
  "id": "svalbard",
  "name": "Svalbard",
  "location": {
    "latitude": 78.22,
    "longitude": 15.65
  },
  "instruments": [
    {
      "kind": "thermometer",
      "installed": "2020-08-20",
      "last_calibrated": "2025-06-30"
    },
    {
      "kind": "rain gauge",
      "installed": "2020-08-20",
      "last_calibrated": null
    }
  ],
  "status": {
    "active": false,
    "issues": [
      {
        "since": "2026-01-12",
        "summary": "rain gauge buried in snow"
      }
    ]
  }
}


The output is JSON again, so `null` and `false` appear as JSON writes them. Svalbard's location has
two members where Bergen's has three, one instrument has never been calibrated, and the status holds
a list of issues. That is one station's shape, though, and laying out forty stations to compare
them would take pages.

### An outline of every path

`paths` walks a JSON value and yields the path and type of the value and of everything inside it. It
writes every item of a list as `[*]`, as JSONPath does, so all four stations share one set of paths,
and `Counter` counts how often each path appears with each type:


In [5]:
def paths(value, path="$"):
    """Yield the path and type name of a JSON value, and of everything inside it."""
    yield path, type(value).__name__
    if isinstance(value, dict):
        for key, item in value.items():
            yield from paths(item, f"{path}.{key}")
    elif isinstance(value, list):
        for item in value:
            yield from paths(item, f"{path}[*]")


for (path, kind), count in sorted(Counter(paths(network)).items()):
    print(f"{path:<46} {kind:<8} {count}")


$                                              dict     1
$.name                                         str      1
$.stations                                     list     1
$.stations[*]                                  dict     4
$.stations[*].id                               str      4
$.stations[*].instruments                      list     4
$.stations[*].instruments[*]                   dict     9
$.stations[*].instruments[*].installed         str      9
$.stations[*].instruments[*].kind              str      9
$.stations[*].instruments[*].last_calibrated   NoneType 2
$.stations[*].instruments[*].last_calibrated   str      7
$.stations[*].location                         dict     4
$.stations[*].location.elevation_m             int      3
$.stations[*].location.latitude                float    4
$.stations[*].location.longitude               float    4
$.stations[*].name                             str      4
$.stations[*].status                           NoneType 1
$.stations[*].

Read the counts against the four stations and nine instruments. `elevation_m` appears three times,
so one station lacks it. `status` is a `dict` three times and a `NoneType` once. `last_calibrated`
is a `str` for seven instruments and a `NoneType` for two. Those are the three places where code
that reaches in without checking will fail, found before writing any of it.

### Reaching in, one level at a time

A path is a recipe for reaching a value: a name for each object, and a position or a loop for each
list. Here is `$.stations[1].instruments[2].kind` a step at a time, and then as one expression:


In [6]:
stations = network["stations"]
oslo = stations[1]
instruments = oslo["instruments"]
third = instruments[2]

print(third["kind"])
print(network["stations"][1]["instruments"][2]["kind"])


anemometer
anemometer


The one expression is shorter. The steps name each level, and when a lookup fails, the traceback
points at the line of the step that failed. Where the outline says `[*]`, a loop reaches every
item, and a comprehension collects a field from each:


In [7]:
for station in network["stations"]:
    kinds = [instrument["kind"] for instrument in station["instruments"]]
    print(f"{station['name']:<9} {kinds}")


Bergen    ['thermometer', 'rain gauge']
Oslo      ['thermometer', 'rain gauge', 'anemometer']
Svalbard  ['thermometer', 'rain gauge']
Tromso    ['thermometer', 'anemometer']


### Finding an item by its id

`stations[1]` is Oslo only while Oslo is second in the list, and an API is free to change the order.
An id does not change, so find items by id. `next` returns the first item a generator produces, or
the default it is given when the generator produces none:


In [8]:
svalbard = next((station for station in network["stations"] if station["id"] == "svalbard"), None)
narvik = next((station for station in network["stations"] if station["id"] == "narvik"), None)

print(svalbard["name"], "|", narvik)


Svalbard | None


For more than one lookup, build a dictionary keyed by id once:


In [9]:
by_id = {station["id"]: station for station in network["stations"]}

print(list(by_id))
print(by_id["tromso"]["location"])


['bergen', 'oslo', 'svalbard', 'tromso']
{'latitude': 69.65, 'longitude': 18.96, 'elevation_m': 100}


### A field that is missing, or null

`get` returns a key's value when the key is there, and `None`, or the default it is given, when the
key is not:


In [10]:
for station in network["stations"]:
    location = station["location"]
    print(f"{station['name']:<9} {location.get('elevation_m')!r:<5} {location.get('elevation_m', 'unknown')!r}")


Bergen    12    12
Oslo      94    94
Svalbard  None  'unknown'
Tromso    100   100


The default applies only when the key is missing. A key that is present with the value `null` gives
`None`, whatever the default:


In [11]:
rain_gauge = by_id["oslo"]["instruments"][1]

print(rain_gauge)
print("get with a default:", rain_gauge.get("last_calibrated", "never"))
print("key present:       ", "last_calibrated" in rain_gauge)


{'kind': 'rain gauge', 'installed': '2016-03-15', 'last_calibrated': None}
get with a default: None
key present:        True


`in` tells the two apart: the key is there, and its value is `None`. When missing and `null` mean the
same thing to the program, as they usually do, read the value with `get` and test it with `is None`,
which covers both. `or` is shorter, as in `value or "never"`, but it also replaces a value that is
present and false: `0`, an empty string, `False` or an empty list.

### Every level of a path: dig

A path several levels deep can stop at any level: a key missing, a position past the end of a list,
or an object that is `null`. `dig` follows a path of names and positions, and returns a default at
the first level that is not there:


In [12]:
def dig(value, *path, default=None):
    """Follow names and positions into JSON, returning default where a level is missing or null."""
    for step in path:
        if value is None:
            return default
        try:
            value = value[step]
        except (KeyError, IndexError):
            return default
    return default if value is None else value


for station in network["stations"]:
    active = dig(station, "status", "active", default="not reported")
    print(f"{station['name']:<9} active={active!s:<13} first issue={dig(station, 'status', 'issues', 0, 'summary')}")


Bergen    active=True          first issue=None
Oslo      active=True          first issue=None
Svalbard  active=False         first issue=rain gauge buried in snow
Tromso    active=not reported  first issue=None


Tromso's `status` is `null`, so `dig` stopped there and returned the default. Bergen and Oslo have no
issues, so position `0` was past the end of an empty list. `dig` still raises when a step cannot
apply to the value at all, such as a name used on a list, because then the path is wrong rather than
the data incomplete.

### Dates in a document

JSON has no date type, so `updated` and every date of installation and calibration arrived as a
string. `datetime.fromisoformat` and `date.fromisoformat` parse the ISO 8601 forms APIs send, the
`Z` that stands for UTC included:


In [13]:
updated = datetime.fromisoformat(network["updated"])
installed = date.fromisoformat(by_id["oslo"]["instruments"][2]["installed"])

print(repr(network["updated"]), "->", repr(updated))
print(repr(by_id["oslo"]["instruments"][2]["installed"]), "->", repr(installed))
print("installed", (updated.date() - installed).days, "days before the update")


'2026-03-01T09:00:00Z' -> datetime.datetime(2026, 3, 1, 9, 0, tzinfo=datetime.timezone.utc)
'2022-09-01' -> datetime.date(2022, 9, 1)
installed 1277 days before the update


Parsed, the dates can be subtracted, which the report at the end of this section relies on. The
**JSON on Disk** notebook went the other way, writing dates out as ISO 8601 text. Numbers need no
parsing, as `12` and `60.39` showed. Some APIs send large ids as strings, because JavaScript cannot
hold every integer above `2**53` exactly, so an API's documentation is the place to find whether an
id is a number or text.

### Columns to rows: zip

Open-Meteo's `daily` is shaped differently from the network document. The network sends rows, an
object for each station; `daily` sends columns, a list for each variable, matched by position. The
**What an API Is** notebook paired two of those columns with `zip`. Here are four, from the request
the **Query Parameters** notebook sent for Tromso:


In [14]:
weather = requests.get(OPEN_METEO, timeout=30, params={
    "latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15", "end_date": "2025-01-17",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum", "models": "era5"}).json()

daily = weather["daily"]
for name, column in daily.items():
    print(f"{name:<19} {column}")


time                ['2025-01-15', '2025-01-16', '2025-01-17']
temperature_2m_max  [6.5, 7.1, 8.6]
temperature_2m_min  [2.7, 5.8, 6.2]
precipitation_sum   [23.7, 21.1, 18.8]


`zip` takes any number of lists, so `zip(*daily.values())` pairs the first value of every column,
then the second, then the third. `dict(zip(daily, row))` puts the names back on:


In [15]:
rows = [dict(zip(daily, row)) for row in zip(*daily.values())]

for row in rows:
    print(row)


{'time': '2025-01-15', 'temperature_2m_max': 6.5, 'temperature_2m_min': 2.7, 'precipitation_sum': 23.7}
{'time': '2025-01-16', 'temperature_2m_max': 7.1, 'temperature_2m_min': 5.8, 'precipitation_sum': 21.1}
{'time': '2025-01-17', 'temperature_2m_max': 8.6, 'temperature_2m_min': 6.2, 'precipitation_sum': 18.8}


Each row is an object holding every variable for one day, the shape the network document uses for
its stations. Rows are the easier shape to filter and sort, as in finding the wettest day:


In [16]:
wettest = max(rows, key=lambda row: row["precipitation_sum"])

print(wettest["time"], wettest["precipitation_sum"], weather["daily_units"]["precipitation_sum"])


2025-01-15 23.7 mm


Columns are the smaller shape: each name appears once, however many days are asked for. The **Pandas, Deep Dive** guide reads either shape straight into a table.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.

### A calibration report for the network

Everything in this notebook, in one report. For each station, it prints the elevation or says it is
unknown, and the status or says it was not reported, with any issues. For each instrument, it
prints how many days before the document's update it was calibrated, or that it never was, and marks
an instrument overdue after a year.


In [17]:
def report(network):
    """Print each station's state and each instrument's calibration, allowing for what is absent."""
    updated = datetime.fromisoformat(network["updated"]).date()
    for station in network["stations"]:
        elevation = dig(station, "location", "elevation_m")
        where = "elevation unknown" if elevation is None else f"{elevation} m"
        status = station["status"]
        if status is None:
            state = "status not reported"
        elif status["active"]:
            state = "active"
        else:
            state = "inactive: " + "; ".join(issue["summary"] for issue in status["issues"])
        print(f"{station['name']}, {where}, {state}")

        for instrument in station["instruments"]:
            calibrated = instrument["last_calibrated"]
            if calibrated is None:
                note = "never calibrated"
            else:
                days = (updated - date.fromisoformat(calibrated)).days
                note = f"calibrated {days} days before the update" + (", overdue" if days > 365 else "")
            print(f"    {instrument['kind']:<12} {note}")


report(network)


Bergen, 12 m, active
    thermometer  calibrated 138 days before the update
    rain gauge   calibrated 138 days before the update
Oslo, 94 m, active
    thermometer  calibrated 89 days before the update
    rain gauge   never calibrated
    anemometer   calibrated 466 days before the update, overdue
Svalbard, elevation unknown, inactive: rain gauge buried in snow
    thermometer  calibrated 244 days before the update
    rain gauge   never calibrated
Tromso, 100 m, status not reported
    thermometer  calibrated 40 days before the update
    anemometer   calibrated 383 days before the update, overdue


### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `network["stations"]`, then `station["instruments"]` | a name for each object, a loop for each list | Reaching in, one level at a time |
| `dig(station, "location", "elevation_m")` | a field that one station lacks | Every level of a path: dig |
| `if status is None` | an object that may be `null` | A field that is missing, or null |
| `if calibrated is None` | a value that is present, and `null` | A field that is missing, or null |
| `datetime.fromisoformat`, `date.fromisoformat` | dates that arrive as text | Dates in a document |

Each check for a gap answers a line of the outline, which counted `elevation_m` three times among
four stations, `status` as `NoneType` once, and `last_calibrated` as `NoneType` twice.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/06-json-in-a-response-solutions.ipynb).

**1.** Print the network's `name` and `updated` members, and the number of instruments across all of
its stations.


In [18]:
# your code here


**2.** Find Tromso with `by_id`, and lay its station out with `json.dumps` and `indent=2`.


In [19]:
# your code here


**3.** Print every instrument that has never been calibrated, in the form `Oslo: rain gauge`.


In [20]:
# your code here


**4.** With a list comprehension, print the ids of the stations whose location has no `elevation_m`.


In [21]:
# your code here


**5.** Ask Open-Meteo for Bergen's daily maximum and minimum temperatures and precipitation, for the
same dates as the Tromso request. Turn `daily` into rows, and print the day with the lowest minimum
temperature. Bergen is at latitude `60.39` and longitude `5.32`.


In [22]:
# your code here


**6.** Write `installed_before(network, year)`, which returns a `(station id, instrument kind)` pair
for every instrument installed before the first day of that year. Print what it returns for `2019`.


In [23]:
# your code here


## Common errors

### KeyError: 'elevation_m'


In [24]:
for station in network["stations"]:
    print(station["name"], station["location"]["elevation_m"])


Bergen 12
Oslo 94


KeyError: 'elevation_m'

Two stations printed, and the third, Svalbard, raised: its location has no `elevation_m`. The outline
counted that field three times among four stations, which was the warning. Reach for a field that
may be missing with `get`:


In [25]:
for station in network["stations"]:
    print(station["name"], station["location"].get("elevation_m", "unknown"))


Bergen 12
Oslo 94
Svalbard unknown
Tromso 100


### TypeError: 'NoneType' object is not subscriptable


In [26]:
for station in network["stations"]:
    print(station["name"], station["status"]["active"])


Bergen True
Oslo True
Svalbard False


TypeError: 'NoneType' object is not subscriptable

Three stations printed, and Tromso raised: its `status` is `null`, which arrived as `None`, and
`None` has no members to look up. The message names a type rather than a field, so read the lookup
before the one that failed, `station["status"]`. Test for `None` before reaching past it:


In [27]:
for station in network["stations"]:
    status = station["status"]
    print(station["name"], "not reported" if status is None else status["active"])


Bergen True
Oslo True
Svalbard False
Tromso not reported


### AttributeError: 'NoneType' object has no attribute 'get'


In [28]:
by_id["tromso"].get("status", {}).get("active")


AttributeError: 'NoneType' object has no attribute 'get'

The `{}` guards against a missing `status`, but Tromso's `status` is not missing: it is present,
with the value `null`. `get` returned that `None`, and `None` has no `get`. `or {}` replaces a `None`
as well as a missing key, and `dig` handles both along a whole path:


In [29]:
print((by_id["tromso"].get("status") or {}).get("active"))
print(dig(by_id["tromso"], "status", "active"))


None
None


### TypeError: list indices must be integers or slices, not str


In [30]:
network["stations"]["name"]


TypeError: list indices must be integers or slices, not str

`stations` is a list, `[*]` in the outline, and a list is reached by position or by a loop, not by a
name. Take one item, or collect the field from every item:


In [31]:
print(network["stations"][0]["name"])
print([station["name"] for station in network["stations"]])


Bergen
['Bergen', 'Oslo', 'Svalbard', 'Tromso']


### JSONDecodeError: Extra data: line 2 column 1 (char 334)


In [32]:
export = requests.get(f"{BASE}/network/export", timeout=10)
export.json()


JSONDecodeError: Extra data: line 2 column 1 (char 334)

`/network/export` sends the same stations as JSON Lines, the format the **JSON on Disk** notebook used
for files too big to load at once: one complete JSON value on every line. APIs send it for bulk
exports and streams, so that a client can handle a line at a time. `json()` expects a single value,
so it parsed the first line and then found more text after it, at line 2. The `Content-Type` names
the format, and each line parses on its own:


In [33]:
print(export.headers["Content-Type"])

stations = [json.loads(line) for line in export.text.splitlines()]
print(len(stations), "stations:", [station["id"] for station in stations])


application/x-ndjson
4 stations: ['bergen', 'oslo', 'svalbard', 'tromso']


## Recap

- `response.json()` turns objects into dictionaries, arrays into lists and `null` into `None`.
- Look before reaching: `json.dumps(value, indent=2)` lays one item out, and an outline of paths
  shows which fields every item has, which only some have, and which can be `null`.
- Reach in a level at a time, a name for each object and a position or a loop for each list, and
  find an item by its id rather than its position.
- `get` with a default covers a missing key but not a key that is `null`; `in` tells the two apart,
  and `dig` covers both along a whole path.
- Dates arrive as text, and `fromisoformat` parses them.
- `zip(*columns)` turns Open-Meteo's columns into rows, an object for each day.


## What is next

The **Schemas and Validation** notebook. Every check here was written by hand, a field at a time,
after the outline showed what to expect. That notebook writes the expected shape down once, as a
schema, and checks a whole response against it, with an error that names the field that broke the
promise.


---

&#8592; **Previous:** [Query Parameters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/05-query-parameters.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Schemas and Validation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/07-schemas-and-validation.ipynb) &#8594;
